In [ ]:
!pip install holidays

In [14]:
import pandas as pd
import numpy as np
import holidays
import os

# Load and merge electricity and weather dataset

--- 

in this final project we only choose 1 region for its simplicity, so we can focus more on ML principal principals rather than dataset size problem.

CAL (California) regiom is chosen because it least problematic region:
1. small number of balancing authorities
2. small number/percentage of missing values

In [5]:
selected_region = 'CAL'

In [6]:
weather_df = pd.read_csv(f'weatherEDA_outputs/by_region/open_meteo_{selected_region}_2015-07-01_2026-05-06.csv')
weather_df.head()

,temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,timestamp_utc,Region,region_name,weather_city,weather_state,latitude,longitude
0,31.4,31,1.8,0.0,2015-07-01 00:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
1,29.9,35,12.7,0.0,2015-07-01 01:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
2,28.3,39,8.6,0.0,2015-07-01 02:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
3,27.9,39,1.8,0.0,2015-07-01 03:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437
4,27.3,41,2.2,0.0,2015-07-01 04:00:00+00:00,CAL,California,Los Angeles,California,34.0522,-118.2437


In [7]:
electricity_df = pd.read_csv('electricityEDA_outputs/reproducibility/region_hourly_after_causal_imputation.csv')
electricity_df = electricity_df[electricity_df['Region'] == selected_region]
electricity_df.head()

,timestamp_utc,Region,data_date,region_demand_mw,region_demand_forecast_mw,num_ba,num_rows,num_available_ba,num_missing_ba,year,month,hour,day_of_week,target_region_demand_mw,is_target_imputed,imputation_method
0,2015-07-01 08:00:00+00:00,CAL,2015-07-01,38210.0,35264.0,5,5,5,0,2015,7,8,2,38210.0,0,observed
1,2015-07-01 09:00:00+00:00,CAL,2015-07-01,35171.0,32894.0,5,5,5,0,2015,7,9,2,35171.0,0,observed
2,2015-07-01 10:00:00+00:00,CAL,2015-07-01,33243.0,31360.0,5,5,5,0,2015,7,10,2,33243.0,0,observed
3,2015-07-01 11:00:00+00:00,CAL,2015-07-01,31955.0,30579.0,5,5,5,0,2015,7,11,2,31955.0,0,observed
4,2015-07-01 12:00:00+00:00,CAL,2015-07-01,31199.0,30723.0,5,5,5,0,2015,7,12,2,31199.0,0,observed


# Merging Process

In [9]:
print("Weather DataFrame columns:", weather_df.columns)
print("Electricity DataFrame columns:", electricity_df.columns)

Weather DataFrame columns: Index(['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
       'precipitation', 'timestamp_utc', 'Region', 'region_name',
       'weather_city', 'weather_state', 'latitude', 'longitude'],
      dtype='object')
Electricity DataFrame columns: Index(['timestamp_utc', 'Region', 'data_date', 'region_demand_mw',
       'region_demand_forecast_mw', 'num_ba', 'num_rows', 'num_available_ba',
       'num_missing_ba', 'year', 'month', 'hour', 'day_of_week',
       'target_region_demand_mw', 'is_target_imputed', 'imputation_method'],
      dtype='object')


In [8]:
print("Weather DataFrame shape:", weather_df.shape)
print("Electricity DataFrame shape:", electricity_df.shape)

Weather DataFrame shape: (95112, 11)
Electricity DataFrame shape: (95112, 16)


In [17]:
min_weather = weather_df[weather_df['timestamp_utc'] == weather_df['timestamp_utc'].min()]['timestamp_utc'].values[0]
max_weather = weather_df[weather_df['timestamp_utc'] == weather_df['timestamp_utc'].max()]['timestamp_utc'].values[0]

min_electricity = electricity_df[electricity_df['timestamp_utc'] == electricity_df['timestamp_utc'].min()]['timestamp_utc'].values[0]
max_electricity = electricity_df[electricity_df['timestamp_utc'] == electricity_df['timestamp_utc'].max()]['timestamp_utc'].values[0]

print(f"Weather DataFrame timestamp range: \n{min_weather} to {max_weather}")
print(f"Electricity DataFrame timestamp range: \n{min_electricity} to {max_electricity}")

Weather DataFrame timestamp range: 
2015-07-01 00:00:00+00:00 to 2026-05-06 23:00:00+00:00
Electricity DataFrame timestamp range: 
2015-07-01 08:00:00+00:00 to 2026-05-07 07:00:00+00:00


In [19]:
# clip data time range to be the same
start = max(min_electricity, min_weather)
end   = min(max_electricity, max_weather)

print(f"Clipped timestamp range: \n{start} to {end}")

Clipped timestamp range: 
2015-07-01 08:00:00+00:00 to 2026-05-06 23:00:00+00:00


In [20]:
# Clip both datasets to the overlapping timestamp range
weather_clipped = weather_df[
    (weather_df["timestamp_utc"] >= start) &
    (weather_df["timestamp_utc"] <= end)
].copy()

electricity_clipped = electricity_df[
    (electricity_df["timestamp_utc"] >= start) &
    (electricity_df["timestamp_utc"] <= end)
].copy()

print("Weather clipped shape:", weather_clipped.shape)
print("Electricity clipped shape:", electricity_clipped.shape)

print("Weather clipped range:")
print(weather_clipped["timestamp_utc"].min(), "to", weather_clipped["timestamp_utc"].max())

print("Electricity clipped range:")
print(electricity_clipped["timestamp_utc"].min(), "to", electricity_clipped["timestamp_utc"].max())

# Merge electricity and weather
merged_df = electricity_clipped.merge(
    weather_clipped,
    on=["Region", "timestamp_utc"],
    how="left",
    suffixes=("", "_weather")
)

print("Merged shape:", merged_df.shape)

merged_df.head()

Weather clipped shape: (95104, 11)
Electricity clipped shape: (95104, 16)
Weather clipped range:
2015-07-01 08:00:00+00:00 to 2026-05-06 23:00:00+00:00
Electricity clipped range:
2015-07-01 08:00:00+00:00 to 2026-05-06 23:00:00+00:00
Merged shape: (95104, 25)


,timestamp_utc,Region,data_date,region_demand_mw,region_demand_forecast_mw,num_ba,num_rows,num_available_ba,num_missing_ba,year,...,imputation_method,temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation,region_name,weather_city,weather_state,latitude,longitude
0,2015-07-01 08:00:00+00:00,CAL,2015-07-01,38210.0,35264.0,5,5,5,0,2015,...,observed,23.5,65,4.7,0.0,California,Los Angeles,California,34.0522,-118.2437
1,2015-07-01 09:00:00+00:00,CAL,2015-07-01,35171.0,32894.0,5,5,5,0,2015,...,observed,22.6,71,1.8,0.0,California,Los Angeles,California,34.0522,-118.2437
2,2015-07-01 10:00:00+00:00,CAL,2015-07-01,33243.0,31360.0,5,5,5,0,2015,...,observed,21.7,75,3.9,0.0,California,Los Angeles,California,34.0522,-118.2437
3,2015-07-01 11:00:00+00:00,CAL,2015-07-01,31955.0,30579.0,5,5,5,0,2015,...,observed,21.3,77,4.8,0.0,California,Los Angeles,California,34.0522,-118.2437
4,2015-07-01 12:00:00+00:00,CAL,2015-07-01,31199.0,30723.0,5,5,5,0,2015,...,observed,21.2,77,4.8,0.0,California,Los Angeles,California,34.0522,-118.2437


In [21]:
merged_df.to_csv('merge_outputs/merged_data.csv', index=False)

# Feature engineering (calendar features)

In [3]:
merge_df = pd.read_csv('merge_outputs/merged_data.csv')
merge_df.columns

Index(['timestamp_utc', 'Region', 'data_date', 'region_demand_mw',
       'region_demand_forecast_mw', 'num_ba', 'num_rows', 'num_available_ba',
       'num_missing_ba', 'year', 'month', 'hour', 'day_of_week',
       'target_region_demand_mw', 'is_target_imputed', 'imputation_method',
       'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m',
       'precipitation', 'region_name', 'weather_city', 'weather_state',
       'latitude', 'longitude'],
      dtype='object')

In [4]:
merge_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95104 entries, 0 to 95103
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   timestamp_utc              95104 non-null  object 
 1   Region                     95104 non-null  object 
 2   data_date                  95104 non-null  object 
 3   region_demand_mw           95088 non-null  float64
 4   region_demand_forecast_mw  95104 non-null  float64
 5   num_ba                     95104 non-null  int64  
 6   num_rows                   95104 non-null  int64  
 7   num_available_ba           95104 non-null  int64  
 8   num_missing_ba             95104 non-null  int64  
 9   year                       95104 non-null  int64  
 10  month                      95104 non-null  int64  
 11  hour                       95104 non-null  int64  
 12  day_of_week                95104 non-null  int64  
 13  target_region_demand_mw    95088 non-null  flo

In [15]:
df_fe = merge_df.copy()

# Convert timestamp columns
df_fe["timestamp_utc"] = pd.to_datetime(df_fe["timestamp_utc"], utc=True)
df_fe["data_date"] = pd.to_datetime(df_fe["data_date"], errors="coerce")

# Use local timezone based on the selected weather/electricity region.
# Current dataset uses CAL with Los Angeles weather proxy.
LOCAL_TIMEZONE = "America/Los_Angeles"

df_fe["timestamp_local"] = df_fe["timestamp_utc"].dt.tz_convert(LOCAL_TIMEZONE)

# Calendar features from local timestamp
df_fe["local_date"] = pd.to_datetime(df_fe["timestamp_local"].dt.date)
df_fe["local_year"] = df_fe["timestamp_local"].dt.year
df_fe["local_month"] = df_fe["timestamp_local"].dt.month
df_fe["local_day"] = df_fe["timestamp_local"].dt.day
df_fe["local_hour"] = df_fe["timestamp_local"].dt.hour
df_fe["local_day_of_week"] = df_fe["timestamp_local"].dt.dayofweek  # Monday=0, Sunday=6
df_fe["local_day_name"] = df_fe["timestamp_local"].dt.day_name()

# Weekend feature
df_fe["is_weekend"] = df_fe["local_day_of_week"].isin([5, 6]).astype(int)

# Cyclical calendar features
df_fe["hour_sin"] = np.sin(2 * np.pi * df_fe["local_hour"] / 24)
df_fe["hour_cos"] = np.cos(2 * np.pi * df_fe["local_hour"] / 24)

df_fe["day_of_week_sin"] = np.sin(2 * np.pi * df_fe["local_day_of_week"] / 7)
df_fe["day_of_week_cos"] = np.cos(2 * np.pi * df_fe["local_day_of_week"] / 7)

df_fe["month_sin"] = np.sin(2 * np.pi * df_fe["local_month"] / 12)
df_fe["month_cos"] = np.cos(2 * np.pi * df_fe["local_month"] / 12)

# Season feature for EDA or categorical modeling
season_map = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "fall", 10: "fall", 11: "fall"
}

df_fe["season"] = df_fe["local_month"].map(season_map)

# Display selected calendar features
calendar_cols = [
    "timestamp_utc",
    "timestamp_local",
    "data_date",
    "local_date",
    "local_year",
    "local_month",
    "local_day",
    "local_hour",
    "local_day_of_week",
    "local_day_name",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "month_sin",
    "month_cos",
    "season"
]

df_fe[calendar_cols].head(10)

,timestamp_utc,timestamp_local,data_date,local_date,local_year,local_month,local_day,local_hour,local_day_of_week,local_day_name,is_weekend,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,month_sin,month_cos,season
0,2015-07-01 08:00:00+00:00,2015-07-01 01:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,1,2,Wednesday,0,0.258819,9.659258e-01,0.974928,-0.222521,-0.5,-0.866025,summer
1,2015-07-01 09:00:00+00:00,2015-07-01 02:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,2,2,Wednesday,0,0.500000,8.660254e-01,0.974928,-0.222521,-0.5,-0.866025,summer
2,2015-07-01 10:00:00+00:00,2015-07-01 03:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,3,2,Wednesday,0,0.707107,7.071068e-01,0.974928,-0.222521,-0.5,-0.866025,summer
3,2015-07-01 11:00:00+00:00,2015-07-01 04:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,4,2,Wednesday,0,0.866025,5.000000e-01,0.974928,-0.222521,-0.5,-0.866025,summer
4,2015-07-01 12:00:00+00:00,2015-07-01 05:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,5,2,Wednesday,0,0.965926,2.588190e-01,0.974928,-0.222521,-0.5,-0.866025,summer
5,2015-07-01 13:00:00+00:00,2015-07-01 06:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,6,2,Wednesday,0,1.000000,6.123234e-17,0.974928,-0.222521,-0.5,-0.866025,summer
6,2015-07-01 14:00:00+00:00,2015-07-01 07:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,7,2,Wednesday,0,0.965926,-2.588190e-01,0.974928,-0.222521,-0.5,-0.866025,summer
7,2015-07-01 15:00:00+00:00,2015-07-01 08:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,8,2,Wednesday,0,0.866025,-5.000000e-01,0.974928,-0.222521,-0.5,-0.866025,summer
8,2015-07-01 16:00:00+00:00,2015-07-01 09:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,9,2,Wednesday,0,0.707107,-7.071068e-01,0.974928,-0.222521,-0.5,-0.866025,summer
9,2015-07-01 17:00:00+00:00,2015-07-01 10:00:00-07:00,2015-07-01,2015-07-01,2015,7,1,10,2,Wednesday,0,0.500000,-8.660254e-01,0.974928,-0.222521,-0.5,-0.866025,summer


In [17]:
# Annual seasonality feature
df_fe["day_of_year"] = df_fe["timestamp_local"].dt.dayofyear

df_fe["days_in_year"] = np.where(
    df_fe["timestamp_local"].dt.is_leap_year,
    366,
    365
)

df_fe["day_of_year_sin"] = np.sin(
    2 * np.pi * df_fe["day_of_year"] / df_fe["days_in_year"]
)

df_fe["day_of_year_cos"] = np.cos(
    2 * np.pi * df_fe["day_of_year"] / df_fe["days_in_year"]
)

calendar_feature_cols = [
    "hour_sin",
    "hour_cos",
    "day_of_week_sin",
    "day_of_week_cos",
    "is_weekend",
    "day_of_year_sin",
    "day_of_year_cos",
]

df_fe[[
    "timestamp_local",
    "local_hour",
    "local_day_of_week",
    "day_of_year",
    *calendar_feature_cols
]].head()

,timestamp_local,local_hour,local_day_of_week,day_of_year,hour_sin,hour_cos,day_of_week_sin,day_of_week_cos,is_weekend,day_of_year_sin,day_of_year_cos
0,2015-07-01 01:00:00-07:00,1,2,182,0.258819,0.965926,0.974928,-0.222521,0,0.008607,-0.999963
1,2015-07-01 02:00:00-07:00,2,2,182,0.500000,0.866025,0.974928,-0.222521,0,0.008607,-0.999963
2,2015-07-01 03:00:00-07:00,3,2,182,0.707107,0.707107,0.974928,-0.222521,0,0.008607,-0.999963
3,2015-07-01 04:00:00-07:00,4,2,182,0.866025,0.500000,0.974928,-0.222521,0,0.008607,-0.999963
4,2015-07-01 05:00:00-07:00,5,2,182,0.965926,0.258819,0.974928,-0.222521,0,0.008607,-0.999963


In [18]:
# Use US federal holidays only
us_holidays = holidays.US(
    years=range(df_fe["local_year"].min(), df_fe["local_year"].max() + 1)
)

# Make sure local_date is date object
df_fe["local_date"] = pd.to_datetime(df_fe["local_date"]).dt.date

holiday_dates = set(us_holidays.keys())

day_before_holiday_dates = {
    holiday_date - pd.Timedelta(days=1)
    for holiday_date in holiday_dates
}

day_after_holiday_dates = {
    holiday_date + pd.Timedelta(days=1)
    for holiday_date in holiday_dates
}

df_fe["is_holiday"] = df_fe["local_date"].isin(holiday_dates).astype(int)
df_fe["is_day_before_holiday"] = df_fe["local_date"].isin(day_before_holiday_dates).astype(int)
df_fe["is_day_after_holiday"] = df_fe["local_date"].isin(day_after_holiday_dates).astype(int)

df_fe["is_holiday_period"] = (
    (df_fe["is_holiday"] == 1) |
    (df_fe["is_day_before_holiday"] == 1) |
    (df_fe["is_day_after_holiday"] == 1)
).astype(int)

holiday_feature_cols = [
    "is_holiday",
    "is_day_before_holiday",
    "is_day_after_holiday",
    "is_holiday_period"
]

df_fe[[
    "timestamp_local",
    "local_date",
    "local_day_name",
    *holiday_feature_cols
]].head(30)

,timestamp_local,local_date,local_day_name,is_holiday,is_day_before_holiday,is_day_after_holiday,is_holiday_period
0,2015-07-01 01:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
1,2015-07-01 02:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
2,2015-07-01 03:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
3,2015-07-01 04:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
4,2015-07-01 05:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
5,2015-07-01 06:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
6,2015-07-01 07:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
7,2015-07-01 08:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
8,2015-07-01 09:00:00-07:00,2015-07-01,Wednesday,0,0,0,0
9,2015-07-01 10:00:00-07:00,2015-07-01,Wednesday,0,0,0,0


In [19]:
df_fe.to_csv('merge_outputs/all_features_all_timerange.csv', index=False)

# Data Splitting

---

* Warm-up / history : 2015-07-01 sampai 2017-06-30
* Train             : 2017-07-01 sampai 2023-12-31
* Validation        : 2024-01-01 sampai 2024-12-31
* Test              : 2025-01-01 sampai 2026-05-06

In [22]:
SPLIT_OUTPUT_DIR = "final_splits"

df_split = df_fe.copy()

df_split["timestamp_utc"] = pd.to_datetime(df_split["timestamp_utc"], utc=True)
df_split["data_date"] = pd.to_datetime(df_split["data_date"], errors="coerce")

missing_data_date = df_split["data_date"].isna().sum()
missing_timestamp = df_split["timestamp_utc"].isna().sum()

print("Missing data_date after conversion:", missing_data_date)
print("Missing timestamp_utc after conversion:", missing_timestamp)

warmup_start_date = pd.Timestamp("2015-07-01")
modeling_start_date = pd.Timestamp("2017-07-01")

train_end_date = pd.Timestamp("2024-01-01")
valid_end_date = pd.Timestamp("2025-01-01")
test_end_date = pd.Timestamp("2026-05-07")  # exclusive

df_split = df_split.sort_values("timestamp_utc").reset_index(drop=True)

warmup_df = df_split[
    (df_split["data_date"] >= warmup_start_date) &
    (df_split["data_date"] < modeling_start_date)
].copy()

train_df = df_split[
    (df_split["data_date"] >= modeling_start_date) &
    (df_split["data_date"] < train_end_date)
].copy()

valid_df = df_split[
    (df_split["data_date"] >= train_end_date) &
    (df_split["data_date"] < valid_end_date)
].copy()

test_df = df_split[
    (df_split["data_date"] >= valid_end_date) &
    (df_split["data_date"] < test_end_date)
].copy()

# Do not drop NA here.
# Keep full timeline for lag/rolling/window feature engineering.
warmup_df = warmup_df.reset_index(drop=True)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

warmup_df.to_csv(f"{SPLIT_OUTPUT_DIR}/warmup_2015H2_2017H1_full.csv", index=False)
train_df.to_csv(f"{SPLIT_OUTPUT_DIR}/train_2017H2_2023H2_full.csv", index=False)
valid_df.to_csv(f"{SPLIT_OUTPUT_DIR}/valid_2024_full.csv", index=False)
test_df.to_csv(f"{SPLIT_OUTPUT_DIR}/test_2025_2026May_full.csv", index=False)

print("Saved split datasets to:", SPLIT_OUTPUT_DIR)
print()
print("Warm-up:", warmup_df.shape, warmup_df["timestamp_utc"].min(), "to", warmup_df["timestamp_utc"].max())
print("Train:", train_df.shape, train_df["timestamp_utc"].min(), "to", train_df["timestamp_utc"].max())
print("Validation:", valid_df.shape, valid_df["timestamp_utc"].min(), "to", valid_df["timestamp_utc"].max())
print("Test:", test_df.shape, test_df["timestamp_utc"].min(), "to", test_df["timestamp_utc"].max())

print()
print("Missing target check:")
print("Warm-up target_region_demand_mw missing:", warmup_df["target_region_demand_mw"].isna().sum())
print("Train target_region_demand_mw missing:", train_df["target_region_demand_mw"].isna().sum())
print("Validation actual region_demand_mw missing:", valid_df["region_demand_mw"].isna().sum())
print("Test actual region_demand_mw missing:", test_df["region_demand_mw"].isna().sum())

Missing data_date after conversion: 0
Missing timestamp_utc after conversion: 0
Saved split datasets to: final_splits

Warm-up: (17544, 49) 2015-07-01 08:00:00+00:00 to 2017-07-01 07:00:00+00:00
Train: (57001, 49) 2017-07-01 08:00:00+00:00 to 2024-01-01 08:00:00+00:00
Validation: (8784, 49) 2024-01-01 09:00:00+00:00 to 2025-01-01 08:00:00+00:00
Test: (11775, 49) 2025-01-01 09:00:00+00:00 to 2026-05-06 23:00:00+00:00

Missing target check:
Warm-up target_region_demand_mw missing: 0
Train target_region_demand_mw missing: 0
Validation actual region_demand_mw missing: 0
Test actual region_demand_mw missing: 16
